# Chẩn đoán độ chính xác tầng kỹ trên GPU (clip 1)

Clip 1 (`EPf9O7UXRoM.mp4`) đã có 23 ảnh Nguyên chấm đúng/sai. Notebook chạy tầng kỹ theo 5 cấu hình trên **cùng các đoạn nghi vấn** (tầng nhanh chạy một lần) rồi chấm từng cấu hình, để biết FP16, gộp crop hay cỡ ảnh làm mất ca đúng.

**Làm lần lượt:**
1. Settings: bật **Internet**, Accelerator **GPU T4 x2**.
2. **Add Input** Dataset có file `EPf9O7UXRoM.mp4` (Private).
3. Repo private thì thêm Secret `GITHUB_TOKEN` như notebook chạy video.
4. Bấm **Run All** (khoảng 20–25 phút). Cuối notebook in bảng chấm; tải `chan-doan.zip` ở tab Output và gửi lại.


In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

BRANCH = "claude/project-thread-g4frkm"  # nhánh có --fp32/--share-iou; đổi thành "main" sau khi merge
REPO = "github.com/vuthanhhnguyenn/httm"
WORK = Path("/kaggle/working")
REPO_DIR = WORK / "httm"
OUT_ROOT = WORK / "chan-doan"

token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # không có secret: clone bằng URL công khai

def sh(cmd, cwd=None, env=None):
    print("$", cmd)
    try:
        subprocess.run(cmd, shell=True, check=True, cwd=cwd, env=env)
    except subprocess.CalledProcessError as exc:
        # Re-raise without the chained traceback so nothing but the command (never the token) is shown.
        raise RuntimeError(f"Lệnh lỗi (mã {exc.returncode}): {cmd}") from None

print("Python", __import__("sys").version.split()[0])
assert __import__("sys").version_info >= (3, 11), "Cần Python 3.11 trở lên"
CLIP = next(Path("/kaggle/input").rglob("EPf9O7UXRoM.mp4"), None)
assert CLIP, "Không thấy EPf9O7UXRoM.mp4 trong các Dataset đã thêm (Add Input)"
print("Clip:", CLIP)
sh("nvidia-smi -L")

In [ ]:
# Clone nhánh và cài những gói còn thiếu (Kaggle đã có torch, opencv, numpy). Repo không đóng gói pip nên
# các script tự thêm packages/*/src vào sys.path; không cần pip install -e.
# Token goes only through environment variables (git's GIT_CONFIG_* ), never the URL, argv or .git/config.
env = dict(os.environ)
if token:
    import base64
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
               GIT_CONFIG_VALUE_0=f"Authorization: Basic {basic}")
if not REPO_DIR.exists():
    sh(f"git clone --depth 1 --branch {BRANCH} https://{REPO}.git {REPO_DIR}", env=env)
else:
    # Chạy lại trong cùng phiên: lấy commit mới nhất của nhánh thay vì dùng bản cũ đã clone.
    sh(f"git fetch --depth 1 origin {BRANCH}", cwd=REPO_DIR, env=env)
    sh("git reset --hard FETCH_HEAD", cwd=REPO_DIR)
sh('pip install -q "ultralytics==8.4.160" pyyaml "pydantic>=2.9,<3.0"')

In [ ]:
# Tải model (kiểm SHA-256 theo models/manifest.yaml)
sh("python scripts/download_model.py", cwd=REPO_DIR)

In [ ]:
# Cấu hình A chạy đủ 3 lượt; B–E dùng lại tầng nhanh của A (cùng đoạn nghi vấn), chỉ đổi tầng kỹ.
import torch

GPUS = torch.cuda.device_count()
DEVICES = "--device 0 --device2 1" if GPUS >= 2 else "--device 0"
CONFIGS = {
    "A_mac_dinh": "",
    "B_fp32": "--fp32",
    "C_khong_gop_crop": "--share-iou 1.1",
    "D_fp32_khong_gop": "--fp32 --share-iou 1.1",
    "E_balanced": "--quality balanced",
}
OUT_ROOT.mkdir(parents=True, exist_ok=True)
errors = {}
for name, flags in CONFIGS.items():
    out = OUT_ROOT / name
    if name != "A_mac_dinh" and (OUT_ROOT / "A_mac_dinh" / "fast").is_dir():
        shutil.rmtree(out, ignore_errors=True)
        out.mkdir(parents=True)
        shutil.copytree(OUT_ROOT / "A_mac_dinh" / "fast", out / "fast")
        shutil.copyfile(OUT_ROOT / "A_mac_dinh" / "tracks.jsonl", out / "tracks.jsonl")
        flags += " --reuse-fast"
    print(f"\n===== {name}: {flags or '(mặc định)'} =====")
    try:
        sh(f'python scripts/process_video.py "{CLIP}" --out "{out}" {DEVICES} {flags}', cwd=REPO_DIR)
    except Exception as exc:  # noqa: BLE001 - keep going with the next config
        errors[name] = str(exc)
        print(f"LỖI {name}: {exc}")


In [ ]:
# Chấm từng cấu hình theo 23 ảnh đã đánh giá của clip 1
import sys

for src in ("packages/proctoring_ai/src", "packages/proctoring_core/src"):
    if str(REPO_DIR / src) not in sys.path:
        sys.path.insert(0, str(REPO_DIR / src))
from proctoring_ai.multi.cctv.review_score import load_verdicts, score_review  # noqa: E402

verdicts = load_verdicts(REPO_DIR / "data/labels/EPf9O7UXRoM-review.yaml")
rows = []
for name in CONFIGS:
    out = OUT_ROOT / name
    row = {"cấu hình": name, "lỗi": errors.get(name, "")}
    try:
        records = [json.loads(line) for line in (out / "events.jsonl").read_text(encoding="utf-8").splitlines()
                   if line.strip()]
        timing = json.loads((out / "timing.json").read_text(encoding="utf-8"))
        score = score_review(records, verdicts)
        row.update({"tầng kỹ (s)": timing["review_s"], "crop": timing["deep_stats"].get("crops"),
                    "đúng (9)": score["true_covered"], "đỏ đúng": score["true_red"],
                    "đỏ nhầm": score["false_red"], "nhầm mới": score["false_opened"],
                    "sót": ", ".join(score["missed"])})
    except Exception as exc:  # noqa: BLE001
        row["lỗi"] = row["lỗi"] or str(exc)
    rows.append(row)
try:
    import pandas as pd
    pd.set_option("display.width", 250)
    pd.set_option("display.max_colwidth", 120)
    print(pd.DataFrame(rows).to_string(index=False))
except ImportError:
    for r in rows:
        print(r)


In [ ]:
# Nén phần nhỏ cần gửi lại (không kèm video): sự kiện, thời gian, cache tầng kỹ
pack = WORK / "chan-doan-gui"
shutil.rmtree(pack, ignore_errors=True)
for name in CONFIGS:
    src, dst = OUT_ROOT / name, pack / name
    dst.mkdir(parents=True, exist_ok=True)
    for file in ("events.jsonl", "timing.json", "suspects.jsonl", "review_cache.json.gz"):
        if (src / file).is_file():
            shutil.copyfile(src / file, dst / file)
archive = shutil.make_archive(str(WORK / "chan-doan"), "zip", root_dir=WORK, base_dir="chan-doan-gui")
print(f"Đã nén: {archive} ({Path(archive).stat().st_size / 1e6:.1f} MB). Tải ở tab Output rồi gửi lại.")
